<a href="https://colab.research.google.com/github/Sahab00/AI-Powered-Health-Monitoring-of-Hive-Royalty/blob/main/data_annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data annotation

In [ ]:
import os
import base64
from inference_sdk import InferenceHTTPClient

# ROBFLOW CLIENT

client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="fyyvILMNQ3DWx4ZuCkxu"


# PATHS

base_path = "/content/my_dataset"
images_dir = f"{base_path}/images"
labels_dir = f"{base_path}/labels"

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

input_folder = "/content/my_dataset/input_images"

class_names = []


# PROCESS IMAGES

for filename in os.listdir(input_folder):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(input_folder, filename)

    result = client.run_workflow(
        workspace_name="my-dreams",
        workflow_id="detect-count-and-visualize-7",
        images={"image": img_path},
        use_cache=True
    )

    # Save original image
    os.system(f"cp '{img_path}' '{images_dir}/'")

    # Save polygon-annotated image
    output_image_b64 = result[0]["output_image"]
    img_bytes = base64.b64decode(output_image_b64)
    annotated_img_path = os.path.join(
        images_dir, filename.replace(".jpg", "_result.jpg")
    )

    with open(annotated_img_path, "wb") as f:
        f.write(img_bytes)


    # SEGMENTATION LABELS

    preds = result[0]["predictions"]["predictions"]
    img_info = result[0]["predictions"]["image"]
    img_w, img_h = img_info["width"], img_info["height"]

    seg_lines = []

    for ann in preds:
        label = ann["class"]
        if label not in class_names:
            class_names.append(label)

        class_id = class_names.index(label)

        polygon_coords = []
        for p in ann["points"]:
            polygon_coords.append(f"{p['x']/img_w:.6f}")
            polygon_coords.append(f"{p['y']/img_h:.6f}")

        # YOLOv11-seg line
        seg_lines.append(
            f"{class_id} " + " ".join(polygon_coords)
        )

    label_path = os.path.join(
        labels_dir, filename.replace(".jpg", ".txt")
    )

    with open(label_path, "w") as f:
        f.write("\n".join(seg_lines))



In [ ]:
# DATA.YAML

yaml_path = os.path.join(base_path, "data.yaml")
with open(yaml_path, "w") as f:
    f.write("train: images\n")
    f.write("val: images\n\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(" Dataset ready in YOLOv11-seg polygon format!")

In [ ]:
#zipping the dataset
import shutil
from google.colab import files

zip_path = "/content/my_dataset.zip"
shutil.make_archive("/content/my_dataset", 'zip', "/content/my_dataset")

print("✅ Dataset zipped successfully:", zip_path)